In [1]:
node_properties_query = """
CALL db.schema.nodeTypeProperties() 
YIELD nodeLabels, propertyName
WITH nodeLabels[0] AS label, collect(propertyName) AS props
RETURN {labels: label, properties: props} AS output
"""

rel_properties_query = """
CALL db.schema.relTypeProperties() 
YIELD relType, propertyName
WITH relType, collect(propertyName) AS props
RETURN {type: relType, properties: props} AS output
"""

rel_query = """
CALL db.schema.visualization() 
YIELD nodes, relationships
UNWIND relationships AS rel
WITH startNode(rel) AS source, endNode(rel) AS target, type(rel) AS relType
RETURN {source: labels(source)[0], relationship: relType, target: labels(target)[0]} AS output
"""

In [2]:
from neo4j import GraphDatabase
from neo4j.exceptions import CypherSyntaxError
from openai import OpenAI

def schema_text(node_props, rel_props, rels):
    return f"""
  Đây là schema của cơ sở dữ liệu Neo4j.
  Các thuộc tính của node bao gồm:
  {node_props}
  Các thuộc tính quan hệ (relationship) bao gồm:
  {rel_props}
  Hướng quan hệ từ source node đến target nodes như sau:
  {rels}
  Hãy đảm bảo tuân thủ đúng các loại quan hệ và hướng của chúng.
  """

class Neo4jGeminiQuery:
    def __init__(self, url, user, password, gemini_api_key):
        self.driver = GraphDatabase.driver(url, auth=(user, password))
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=gemini_api_key
        )
        # construct schema
        self.schema = self.generate_schema()


    def generate_schema(self):
        node_props = self.query_database(node_properties_query)
        rel_props = self.query_database(rel_properties_query)
        rels = self.query_database(rel_query)
        return schema_text(node_props, rel_props, rels)

    def refresh_schema(self):
        self.schema = self.generate_schema()

    def get_rewrite_message(self):
        return f"""
        Bạn là chuyên gia xử lý ngôn ngữ tự nhiên cho các câu hỏi truy vấn đến cơ sở dữ liệu đồ thị Neo4j.
        Nhiệm vụ: Dựa vào câu hỏi dưới dạng ngôn ngữ tự nhiên cho người dùng, hãy viết lại câu hỏi thành câu truy vấn dưới dạng ngôn ngữ tự nhiên dựa theo schema được định nghĩa mà từ câu truy vấn đó có thể chuyển sang câu lệnh Cypher một cách chính xác.
        Hướng dẫn các nguyên tắc:
        - Câu truy vấn phải giữ đúng nội dung của câu hỏi, không được tự phát sinh làm thay đổi các thông tin về node và thuộc tính có trong câu hỏi.
        - Trong câu truy vấn, tuyệt đối không được nhắc đến tên loại quan hệ (relationship type) cụ thể.
        - Dựa vào schema, bạn phải sắp xếp lại trật tự trong câu truy vấn để tuân theo hướng của các quan hệ, sao cho source node xuất hiện trước target node.
        - Đối với các thuộc tính dùng để định danh, hãy luôn sử dụng từ "bằng" kết hợp với giá trị tìm kiếm dưới dạng chữ (ví dụ: a, b, c), số (ví dụ: 1, 2, 3) hay chữ số La Mã (ví dụ: I, II, III), đồng thời giữ đúng tên của thuộc tính định danh đó.
        - Đối với các thuộc tính văn bản mô tả, hãy luôn sử dụng từ "chứa" kết hợp với giá trị tìm kiếm.
        Dưới đây là schema của cơ sở dữ liệu Neo4j:
        {self.schema}
        Quy tắc đầu ra:
        - Chỉ trả về duy nhất câu truy vấn hợp lệ dưới dạng ngôn ngữ tự nhiên.
        - Tuyệt đối không thêm bất kỳ văn bản chào hỏi, giải thích, lưu ý hay phân tích nào khác.
        """

    def rewrite_question(self, question):
        messages = [
            {"role": "system", "content": self.get_rewrite_message()},
            {"role": "user", "content": question},
        ]

        completions = self.client.chat.completions.create(
            model="google/gemini-2.5-flash",
            temperature=0.0,
            # max_tokens=1000,
            messages=messages
        )
        return completions.choices[0].message.content

    def get_system_message(self):
        return f"""
        Bạn là chuyên gia viết các câu truy vấn Cypher cho cơ sở dữ liệu đồ thị Neo4j, đóng vai trò tiếp nhận câu hỏi dưới dạng ngôn ngữ tự nhiên và tạo câu lệnh Cypher tương ứng với câu hỏi đó (Text-to-Cypher).
        Nhiệm vụ: Tạo câu lệnh Cypher để truy vấn cơ sở dữ liệu đồ thị Neo4j dựa trên định nghĩa schema được cung cấp.
        Hướng dẫn các nguyên tắc:
        - Phải sử dụng mối quan hệ có độ dài biến đổi (ví dụ: (A)-[*1..]->(B)) nếu xuất hiện bất kỳ đường đi hợp lệ nào có thể kết nối từ source node đến target node theo định nghĩa trong schema. Bạn tuyệt đối không được sử dụng bất kỳ loại quan hệ (relationship type) cụ thể nào ở mệnh đề MATCH.
        - Theo mặc định, không thêm bất kỳ node trung gian nào vào đường đi kết nối các node trong mệnh đề MATCH nếu node trung gian đó không có trên câu hỏi. Tuy nhiên, nếu câu hỏi của người dùng có yêu cầu sử dụng các thuộc tính của node trung gian để lọc hay trả về thì đến lúc đó bạn phải khai báo rõ ràng node trung gian đó ở đường đi trong mệnh để MATCH để áp dụng mệnh đề WHERE hay RETURN một cách chính xác.
        - Tuyệt đối tuân thủ hướng của các mối quan hệ mà chỉ được định nghĩa ở trong schema. Bạn tuyệt đối không được đảo ngược hướng của quan hệ hay đảo target node lên vị trí của source node trong mệnh đề MATCH ở bất kỳ trường hợp nào.
        - Đối với các thuộc tính dùng để định danh (ví dụ: number), hãy luôn sử dụng toán tử =, ngay cả khi giá trị tìm kiếm là chữ cái (ví dụ: a, b, c), số (ví dụ: 1, 2, 3) hay là chữ số La Mã (ví dụ: I, II, III).
        - Đối với các thuộc tính văn bản mô tả (ví dụ: doc_name, title, content), hãy luôn sử dụng toán tử toán tử CONTAINS trong mệnh đề WHERE, đồng thời áp dụng hàm toLower() cho cả thuộc tính văn bản mô tả đó lẫn giá trị tìm kiếm trong mệnh đề WHERE.
        - Không bao giờ trả về toàn bộ node object. Thay vào đó, ở mệnh để RETURN, chỉ chọn các thuộc tính có liên quan đến câu hỏi của người dùng có trong các node này.
        - Nếu không thể tạo ra câu lệnh Cypher từ schema đã cung cấp, hãy giải thích lý do cho người dùng.
        Dưới đây là schema của cơ sở dữ liệu đồ thị Neo4j:
        {self.schema}
        Quy tắc đầu ra:
        - Chỉ trả về duy nhất câu lệnh Cypher hợp lệ và tuyệt đối không được bọc trong block code markdown (không dùng ```cypher ```).
        - Tuyệt đối không thêm bất kỳ văn bản chào hỏi, giải thích, lưu ý hay phân tích nào khác.
        - Trường hợp duy nhất được phép trả về văn bản mà không có Cypher query là khi không thể tạo được câu lệnh từ schema, lúc đó hãy đưa ra nguyên nhân trực tiếp.
        Ví dụ:
        Input: Ai là người ký Luật đường bộ?
        Output: MATCH (d:Document)-[*1..]->(s:Signer) WHERE toLower(d.doc_name) CONTAINS toLower('Luật đường bộ') RETURN s.name
        Input: Nghị định 168 có bao nhiêu điều?
        Output: MATCH (d:Document)-[*1..]->(a:Article) WHERE toLower(d.doc_name) CONTAINS toLower('Nghị định 168') RETURN count(a)
        Input: Văn bản nào có chứa quy định về việc tịch thu phương tiện?
        Output: MATCH (d:Document)-[*1..]->(child) WHERE child:Section OR child:Part OR child:Chapter OR child:Article OR child:Clause OR child:Point WITH d, child WHERE toLower(child.title) CONTAINS "tịch thu phương tiện" OR toLower(child.content) CONTAINS "tịch thu phương tiện" RETURN DISTINCT d.doc_name

        Lưu ý: Không bao gồm bất kỳ lời giải thích hay xin lỗi nào trong câu trả lời của bạn.
        """

    def query_database(self, neo4j_query, params={}):
        with self.driver.session() as session:
            result = session.run(neo4j_query, params)
            output = [r.values() for r in result]
            output.insert(0, result.keys())
            return output

    def construct_cypher(self, question, rewrite, history=None):
        messages = [
            {"role": "system", "content": self.get_system_message()},
            {"role": "user", "content": f"""Câu hỏi gốc của người dùng: {question}.
                                Câu truy vấn đã được viết lại: {rewrite}."""},
        ]
        # Used for Cypher healing flows
        if history:
            messages.extend(history)

        completions = self.client.chat.completions.create(
            model="google/gemini-2.5-flash",
            temperature=0.0,
            # max_tokens=1000,
            messages=messages
        )
        return completions.choices[0].message.content

    def run(self, question, history=None, retry=True):
        # Construct Cypher statement
        rewrite = self.rewrite_question(question)
        print(rewrite)
        cypher = self.construct_cypher(question, rewrite, history)
        print(cypher)
        try:
            return self.query_database(cypher)
        # Self-healing flow
        except CypherSyntaxError as e:
            # If out of retries
            if not retry:
              return "Cú pháp Cypher không hợp lệ."
        # Self-healing Cypher flow by
        # providing specific error to Gemini
            print("Đang thử lại")
            return self.run(
                question,
                [
                    {"role": "assistant", "content": cypher},
                    {
                        "role": "user",
                        "content": f"""Câu truy vấn này trả về lỗi sau: {str(e)} 
                        Hãy cung cấp cho tôi một câu truy vấn đã được cải thiện để chạy được mà không cần bất kỳ lời giải thích hay xin lỗi nào.""",
                    },
                ],
                retry=False
            )

In [3]:
import os
api_key = os.environ.get("OPENROUTER_API_KEY")

In [4]:
gds_db = Neo4jGeminiQuery(
    url=os.environ.get("NEO4J_URI"),
    user=os.environ.get("NEO4J_USER"),
    password=os.environ.get("NEO4J_PASSWORD"),
    gemini_api_key=api_key,
)

Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nCALL db.schema.nodeTypeProperties() \nYIELD nodeLabels, propertyName\nWITH nodeLabels[0] AS label, collect(propertyName) AS props\nRETURN {labels: label, properties: props} AS output\n'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62',

In [5]:
gds_db.run("""
Trần Thanh Mẫn đã ký những văn bản nào?
""")

Tìm các Document được ký bởi Signer có tên bằng Trần Thanh Mẫn.
MATCH (d:Document)-[:SIGNED_BY]->(s:Signer) WHERE toLower(s.name) CONTAINS toLower('Trần Thanh Mẫn') RETURN d.doc_name


[['d.doc_name'],
 ['Luật Đường bộ của Quốc hội, số 35/2024/QH15'],
 ['Luật Trật tự, an toàn giao thông đường bộ của Quốc hội, số 36/2024/QH15'],
 ['Luật sửa đổi, bổ sung một số điều của Luật Xử lý vi phạm hành chính của Quốc hội, số 88/2025/QH15'],
 ['Luật sửa đổi, bổ sung một số điều của 10 Luật có liên quan đến an ninh, trật tự của Quốc hội, số 118/2025/QH15'],
 ['Luật sửa đổi, bổ sung một số điều của Luật Chứng khoán, Luật Kế toán, Luật Kiểm toán độc lập, Luật Ngân sách Nhà nước, Luật Quản lý, sử dụng tài sản công, Luật Quản lý thuế, Luật Thuế thu nhập cá nhân, Luật Dự trữ quốc gia, Luật Xử lý vi phạm hành chính của Quốc hội, số 56/2024/QH15']]

In [6]:
gds_db.run("""
Liệt kê các văn bản thuộc lĩnh vực Giao thông?
""")

Tìm các Document có Field có tên bằng 'Giao thông'.
MATCH (d:Document)-[*1..]->(f:Field) WHERE toLower(f.name) CONTAINS toLower('Giao thông') RETURN d.doc_name


[['d.doc_name'],
 ['Nghị định 119/2024/NĐ-CP của Chính phủ quy định về thanh toán điện tử giao thông đường bộ'],
 ['Luật Trật tự, an toàn giao thông đường bộ của Quốc hội, số 36/2024/QH15'],
 ['Luật Đường bộ của Quốc hội, số 35/2024/QH15'],
 ['Nghị định 123/2021/NĐ-CP của Chính phủ về việc sửa đổi, bổ sung một số điều của các Nghị định quy định xử phạt vi phạm hành chính trong lĩnh vực hàng hải; giao thông đường bộ, đường sắt; hàng không dân dụng'],
 ['Nghị định 100/2019/NĐ-CP của Chính phủ về việc quy định xử phạt vi phạm hành chính trong lĩnh vực giao thông đường bộ và đường sắt'],
 ['Nghị định 168/2024/NĐ-CP của Chính phủ quy định xử phạt vi phạm hành chính về trật tự, an toàn giao thông trong lĩnh vực giao thông đường bộ; trừ điểm, phục hồi điểm giấy phép lái xe']]

In [7]:
gds_db.run("""
Quốc hội ban hành những văn bản nào?
""")

Tìm các Document được ban hành bởi Organization có tên bằng "Quốc hội"
MATCH (d:Document)-[:ISSUED_BY]->(o:Organization) WHERE toLower(o.name) CONTAINS toLower('Quốc hội') RETURN d.doc_name


[['d.doc_name'],
 ['Luật sửa đổi, bổ sung một số điều của Luật Xử lý vi phạm hành chính của Quốc hội, số 67/2020/QH14'],
 ['Luật Phòng, chống tác hại của rượu, bia của Quốc hội, số 44/2019/QH14'],
 ['Luật Đường bộ của Quốc hội, số 35/2024/QH15'],
 ['Luật Trật tự, an toàn giao thông đường bộ của Quốc hội, số 36/2024/QH15'],
 ['Luật Xử lý vi phạm hành chính của Quốc hội, số 15/2012/QH13'],
 ['Luật sửa đổi, bổ sung một số điều của Luật Xử lý vi phạm hành chính của Quốc hội, số 88/2025/QH15'],
 ['Luật sửa đổi, bổ sung một số điều của 10 Luật có liên quan đến an ninh, trật tự của Quốc hội, số 118/2025/QH15'],
 ['Luật sửa đổi, bổ sung một số điều của Luật Chứng khoán, Luật Kế toán, Luật Kiểm toán độc lập, Luật Ngân sách Nhà nước, Luật Quản lý, sử dụng tài sản công, Luật Quản lý thuế, Luật Thuế thu nhập cá nhân, Luật Dự trữ quốc gia, Luật Xử lý vi phạm hành chính của Quốc hội, số 56/2024/QH15']]

In [8]:
gds_db.run("""
Có bao nhiêu chương trong Nghị định 100?
""")

Tìm số lượng Chapter của Document có doc_name chứa "Nghị định 100"
MATCH (d:Document)-[*1..]->(c:Chapter) WHERE toLower(d.doc_name) CONTAINS toLower('Nghị định 100') RETURN count(c)


[['count(c)'], [5]]

In [9]:
gds_db.run("""
Có bao nhiêu điều trong Nghị định 100?
""")

Đếm
MATCH (d:Document)-[*1..]->(a:Article) WHERE toLower(d.doc_name) CONTAINS toLower('Nghị định 100') RETURN count(a)


[['count(a)'], [86]]

In [10]:
gds_db.run("""
Liệt kê tiêu đề của tất cả các điều thuộc chương I trong Nghị định 168?
""")

Tìm các Article có parent_chapter là Chapter có number bằng "I" và doc_identity là Document có doc_name chứa "Nghị định 168", và trả về title của Article.
MATCH (d:Document)-[*1..]->(c:Chapter)-[*1..]->(a:Article) WHERE toLower(d.doc_name) CONTAINS toLower('Nghị định 168') AND c.number = 'I' RETURN a.title


[['a.title'],
 ['Tước quyền sử dụng giấy phép, chứng chỉ hành nghề có thời hạn'],
 ['Đối tượng áp dụng'],
 ['Thời hiệu xử phạt vi phạm hành chính; hành vi vi phạm hành chính đã kết thúc, hành vi vi phạm hành chính đang thực hiện'],
 ['Hình thức xử phạt vi phạm hành chính, biện pháp khắc phục hậu quả; thu hồi giấy phép, chứng chỉ hành nghề'],
 ['Phạm vi điều chỉnh']]

In [11]:
gds_db.run("""
Trong Nghị định 168, điều 1 có tất cả bao nhiêu khoản và có tất cả bao nhiêu điểm?
""")

Tìm số lượng Clause và số lượng Point của Article có doc_identity bằng "Nghị định 168" và number bằng "1"
MATCH (d:Document)-[*1..]->(a:Article) WHERE toLower(d.doc_name) CONTAINS toLower('Nghị định 168') AND a.number = '1' OPTIONAL MATCH (a)-[:HAS_CLAUSE]->(c:Clause) OPTIONAL MATCH (a)-[:HAS_CLAUSE]->(c_with_points:Clause)-[:HAS_POINT]->(p:Point) RETURN count(DISTINCT c) AS ClauseCount, count(DISTINCT p) AS PointCount


[['ClauseCount', 'PointCount'], [2, 2]]

In [12]:
gds_db.run("""
Cho biết nội dung của khoản 2 điều 1 nghị định 168?
""")

Tìm Khoản có số bằng "2" và thuộc về Điều có số bằng "1" và thuộc về Nghị định có doc_name chứa "168", sau đó trả về nội dung của Khoản đó.
MATCH (d:Document)-[*1..]->(a:Article)-[*1..]->(c:Clause) WHERE toLower(d.doc_name) CONTAINS toLower('Nghị định 168') AND a.number = '1' AND c.number = '2' RETURN c.content


[['c.content'],
 ['Các hành vi vi phạm hành chính trong lĩnh vực quản lý nhà nước khác liên quan đến trật tự, an toàn giao thông trong lĩnh vực giao thông đường bộ mà không quy định tại Nghị định này thì áp dụng quy định tại các Nghị định quy định về xử phạt vi phạm hành chính trong các lĩnh vực đó để xử phạt.']]

In [13]:
gds_db.run("""
Văn bản nào có nhiều Điều nhất trong hệ thống dữ liệu hiện tại?
""")

Tìm Document có số lượng Article lớn nhất.
MATCH (d:Document)-[*1..]->(a:Article) RETURN d.doc_name, count(a) AS ArticleCount ORDER BY ArticleCount DESC LIMIT 1


[['d.doc_name', 'ArticleCount'],
 ['Luật Xử lý vi phạm hành chính của Quốc hội, số 15/2012/QH13', 142]]

In [14]:
gds_db.run("""
Liệt kê tất cả các văn bản ban hành năm 2020?
""")

Tìm tất cả các Document có issue_date chứa "2020"
MATCH (d:Document) WHERE toLower(d.issue_date) CONTAINS toLower("2020") RETURN d.doc_name, d.issue_date


[['d.doc_name', 'd.issue_date'],
 ['Luật sửa đổi, bổ sung một số điều của Luật Xử lý vi phạm hành chính của Quốc hội, số 67/2020/QH14',
  '2020-11-13T00:00:00']]